# RCC Python large-data notebook example

This notebook demonstrates the classroom pattern: inspect a small sample interactively, then move expensive work into a Slurm job. It uses a synthetic dataset and does not contact RCC services.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

rng = np.random.default_rng(42)
n = 100_000
df = pd.DataFrame({
    'patient_group': rng.choice(['A', 'B', 'C'], size=n),
    'measurement': rng.normal(loc=0, scale=1, size=n),
    'batch': rng.integers(1, 9, size=n),
})
df.head()


In [ ]:
summary = df.groupby('patient_group')['measurement'].agg(['count', 'mean', 'std'])
summary


In [ ]:
ax = df.sample(5000, random_state=1).boxplot(column='measurement', by='patient_group')
ax.set_title('Sampled distribution by group')
ax.figure.suptitle('')


## RiboSnake-inspired scientific visuals

The next cells use a small synthetic abundance table to demonstrate two reviewable figures: a Bray–Curtis principal coordinates analysis (PCoA) and a ranked waterfall plot. These are teaching visuals, not biological results. In a real analysis, record the distance metric, transformations, grouping variables, and filtering decisions alongside the figure.


In [ ]:
import matplotlib.pyplot as plt

sample_count, feature_count = 36, 18
sample_group = np.repeat(['Reference', 'Treatment A', 'Treatment B'], sample_count // 3)
abundance = rng.gamma(shape=1.6, scale=35, size=(sample_count, feature_count))
abundance[sample_group == 'Treatment A', :5] *= 2.2
abundance[sample_group == 'Treatment B', 5:10] *= 2.4
relative_abundance = abundance / abundance.sum(axis=1, keepdims=True)

# Bray-Curtis dissimilarity followed by classical multidimensional scaling.
difference = np.abs(relative_abundance[:, None, :] - relative_abundance[None, :, :]).sum(axis=2)
total = (relative_abundance[:, None, :] + relative_abundance[None, :, :]).sum(axis=2)
distance = np.divide(difference, total, out=np.zeros_like(difference), where=total != 0)
centering = np.eye(sample_count) - np.ones((sample_count, sample_count)) / sample_count
gram = -0.5 * centering @ (distance ** 2) @ centering
eigenvalues, eigenvectors = np.linalg.eigh(gram)
order = np.argsort(eigenvalues)[::-1]
eigenvalues, eigenvectors = eigenvalues[order], eigenvectors[:, order]
positive = eigenvalues > 0
coordinates = eigenvectors[:, :2] * np.sqrt(np.maximum(eigenvalues[:2], 0))
explained = 100 * eigenvalues[:2] / eigenvalues[positive].sum()


In [ ]:
fig, ax = plt.subplots(figsize=(8, 6), constrained_layout=True)
styles = {
    'Reference': ('#4477AA', 'o'),
    'Treatment A': ('#EE6677', '^'),
    'Treatment B': ('#228833', 's'),
}
for group, (color, marker) in styles.items():
    selected = sample_group == group
    ax.scatter(coordinates[selected, 0], coordinates[selected, 1],
               label=group, color=color, marker=marker, s=70, alpha=0.85,
               edgecolor='white', linewidth=0.7)
ax.axhline(0, color='0.85', linewidth=1, zorder=0)
ax.axvline(0, color='0.85', linewidth=1, zorder=0)
ax.set(
    title='Synthetic community profiles separate by treatment',
    xlabel=f'PCoA 1 ({explained[0]:.1f}% of positive eigenvalue sum)',
    ylabel=f'PCoA 2 ({explained[1]:.1f}% of positive eigenvalue sum)',
)
ax.legend(title='Sample group', frameon=False)
ax.spines[['top', 'right']].set_visible(False)
plt.show()


A PCoA is descriptive: separation suggests a pattern worth investigating, but it does not establish significance or causality. The waterfall view below makes sample-to-sample variation and a predeclared response threshold visible.


In [ ]:
sample_ids = np.array([f'S{i:02d}' for i in range(1, 25)])
relative_change = np.clip(rng.normal(-8, 22, sample_ids.size), -58, 45)
ranked = np.argsort(relative_change)
ranked_change = relative_change[ranked]
ranked_ids = sample_ids[ranked]
bar_colors = np.where(ranked_change <= -20, '#228833', '#CC6677')

fig, ax = plt.subplots(figsize=(10, 5), constrained_layout=True)
ax.bar(ranked_ids, ranked_change, color=bar_colors, width=0.82)
ax.axhline(0, color='0.25', linewidth=1)
ax.axhline(-20, color='#4477AA', linewidth=1.5, linestyle='--', label='Example response threshold (-20%)')
ax.set(
    title='Ranked synthetic response by sample',
    xlabel='Sample, ordered by relative change',
    ylabel='Relative change from baseline (%)',
)
ax.tick_params(axis='x', labelrotation=60)
ax.legend(frameon=False, loc='lower right')
ax.spines[['top', 'right']].set_visible(False)
plt.show()
